# Proyecto 1 — Dashboard de Ventas y Logística
## Punto 12 — Dashboard Plotly Dash (proceso paralelo)
**Autor:** Michael Hans Evans  
**Input:** `superstore_clean.csv`  
**Herramienta:** Python / Plotly Dash  
**Proceso paralelo de:** Power BI Desktop

---
> Este notebook replica el dashboard ejecutivo construido en Power BI  
> utilizando Python y Plotly Dash — demostrando que el mismo resultado  
> se puede alcanzar con distintas herramientas.  
> El dashboard incluye las mismas 5 páginas y segmentaciones que Power BI.


## 0. Importación de librerías

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output

## 1. Carga de datos y sistema de diseño

In [2]:
import sqlite3
from pathlib import Path

# Resolver ruta a la DB independiente del working directory
db_candidates = [Path('database/superstore_model.db'), Path('../database/superstore_model.db')]
db_path = next(p for p in db_candidates if p.exists())

# mode=ro evita conflictos de lock con OneDrive
conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)

df = pd.read_sql('''
    SELECT fo."Order ID", fo."Customer ID", fo."Product ID",
           fo.Sales, fo.Quantity, fo.Discount, fo.Discount_pct,
           fo.Profit, fo.Profit_Margin_pct, fo.Shipping_Days,
           dt."Order Date", dt."Ship Date", dt.Order_Year, dt.Order_Month, dt.Order_Quarter,
           dc."Customer Name", dcs.Segment,
           dgr.Region, dgs.State, dgc.City,
           dp."Product Name", dcat.Category, dsub."Sub-Category",
           ds."Ship Mode"
    FROM fact_orders fo
    JOIN dim_time dt             ON fo."Order ID" = dt."Order ID"
    JOIN dim_customers dc        ON fo."Customer ID" = dc."Customer ID"
    JOIN dim_customers_segment dcs ON dc.segment_key = dcs.segment_key
    JOIN dim_geography dg        ON fo.geography_key = dg.geography_key
    JOIN dim_geography_region dgr ON dg.region_key = dgr.region_key
    JOIN dim_geography_state dgs ON dg.state_key = dgs.state_key
    JOIN dim_geography_city dgc  ON dg.city_key = dgc.city_key
    JOIN dim_products dp         ON fo."Product ID" = dp."Product ID"
    JOIN dim_products_detail dpd ON fo."Product ID" = dpd."Product ID"
    JOIN dim_categories dcat     ON dpd.categories_key = dcat.categories_key
    JOIN dim_sub_categories dsub ON dpd.sub_categories_key = dsub.sub_categories_key
    JOIN dim_shipment ds         ON fo.shipment_key = ds.shipment_key
''', conn, parse_dates=['Order Date', 'Ship Date'])

conn.close()

print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")

COLORS = {
    'bg_canvas'  : '#2D3E50',
    'header'     : '#2E75B6',
    'bg_card'    : '#FFFFFF',
    'sales'      : '#1F6FEB',
    'profit'     : '#1B3A57',
    'text_title' : '#1A2B3C',
    'text_sec'   : '#595959',
    'alert_red'  : '#C00000',
    'alert_yell' : '#FFD966',
    'alert_green': '#375623',
    'dona_2'     : '#52A8FF',
    'dona_4'     : '#0E4D92',
}

CARD_STYLE = {
    'backgroundColor': COLORS['bg_card'],
    'borderRadius'   : '8px',
    'padding'        : '16px',
    'boxShadow'      : '0 2px 8px rgba(0,0,0,0.08)',
    'border'         : '1px solid #D9D9D9'
}

TITLE_STYLE = {
    'color'     : COLORS['text_title'],
    'fontFamily': 'Segoe UI, Arial, sans-serif',
    'fontWeight': '700',
    'margin'    : '0 0 4px 0'
}

Filas: 9,993  |  Columnas: 24


## 2. Funciones de KPIs y gráficos

In [3]:
def calcular_kpis(data):
    total_sales    = data['Sales'].sum()
    total_profit   = data['Profit'].sum()
    total_orders   = len(data)
    profit_margin  = (total_profit / total_sales * 100) if total_sales > 0 else 0
    avg_ticket     = total_sales / total_orders if total_orders > 0 else 0
    avg_discount   = data['Discount_pct'].mean()
    pct_perdida    = (data['Profit'] < 0).mean() * 100
    total_customers= data['Customer ID'].nunique()
    avg_ship_days  = data['Shipping_Days'].mean()

    return {
        'total_sales'    : total_sales,
        'total_profit'   : total_profit,
        'total_orders'   : total_orders,
        'profit_margin'  : profit_margin,
        'avg_ticket'     : avg_ticket,
        'avg_discount'   : avg_discount,
        'pct_perdida'    : pct_perdida,
        'total_customers': total_customers,
        'avg_ship_days'  : avg_ship_days,
    }

def tarjeta_kpi(titulo, valor, formato='numero', color=None):
    if formato == 'moneda':
        val_str = f"$ {valor:,.0f}"
    elif formato == 'pct':
        val_str = f"{valor:.1f}%"
    elif formato == 'dias':
        val_str = f"{valor:.1f} días"
    else:
        val_str = f"{valor:,.0f}"

    color_val = color or COLORS['sales']

    return html.Div([
        html.P(titulo, style={
            'color'     : COLORS['text_sec'],
            'fontSize'  : '12px',
            'fontFamily': 'Segoe UI, Arial, sans-serif',
            'margin'    : '0 0 4px 0',
            'fontWeight': '500'
        }),
        html.H3(val_str, style={
            'color'     : color_val,
            'fontFamily': 'Segoe UI, Arial, sans-serif',
            'fontSize'  : '24px',
            'fontWeight': '700',
            'margin'    : '0'
        })
    ], style=CARD_STYLE)

## 3. Gráficos — verificación visual

In [4]:
subcat = df.groupby('Sub-Category').agg(
    total_profit=('Profit','sum'),
    total_sales =('Sales','sum')
).reset_index().sort_values('total_profit')

subcat['color'] = subcat['total_profit'].apply(
    lambda x: COLORS['alert_red'] if x < 0 else COLORS['sales'])

fig_h1 = go.Figure(go.Bar(
    x=subcat['total_profit'],
    y=subcat['Sub-Category'],
    orientation='h',
    marker_color=subcat['color'],
    text=subcat['total_profit'].apply(lambda x: f"$ {x:,.0f}"),
    textposition='outside'
))
fig_h1.add_vline(x=0, line_color=COLORS['text_sec'], line_width=1)
fig_h1.update_layout(
    title=dict(text="Profit por Subcategoría", font=dict(color=COLORS['text_title'], size=14)),
    plot_bgcolor=COLORS['bg_canvas'],
    paper_bgcolor=COLORS['bg_card'],
    font=dict(family='Segoe UI', color=COLORS['text_title']),
    height=500,
    margin=dict(l=150, r=80, t=50, b=30),
    xaxis=dict(title="Profit (USD)", gridcolor='#E0E0E0'),
    yaxis=dict(title="")
)
fig_h1.show()

In [5]:
df['rango_descuento'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.20, 0.40, 0.60, 1.0],
    labels=['0% Sin descuento','1-20%','21-40%','41-60%','61-80%']
)

h2 = df.groupby('rango_descuento', observed=True).agg(
    avg_profit  =('Profit','mean'),
    pct_perdida =('Profit', lambda x: (x<0).mean()*100),
    total_ordenes=('Sales','count')
).reset_index()

fig_h2 = make_subplots(specs=[[{"secondary_y": True}]])
fig_h2.add_trace(go.Bar(
    x=h2['rango_descuento'], y=h2['avg_profit'],
    name='Avg Profit',
    marker_color=[COLORS['alert_red'] if v < 0 else COLORS['sales'] 
                  for v in h2['avg_profit']],
    text=h2['avg_profit'].apply(lambda x: f"$ {x:,.1f}"),
    textposition='outside'
), secondary_y=False)
fig_h2.add_trace(go.Scatter(
    x=h2['rango_descuento'], y=h2['pct_perdida'],
    name='% Órdenes con pérdida',
    mode='lines+markers',
    line=dict(color=COLORS['alert_red'], width=2),
    marker=dict(size=8)
), secondary_y=True)
fig_h2.add_hline(y=0, line_color=COLORS['text_sec'], line_width=1)
fig_h2.update_layout(
    title=dict(text="Avg Profit y % pérdidas por rango de descuento",
               font=dict(color=COLORS['text_title'], size=14)),
    plot_bgcolor=COLORS['bg_canvas'],
    paper_bgcolor=COLORS['bg_card'],
    font=dict(family='Segoe UI'),
    height=400,
    legend=dict(orientation='h', y=-0.2)
)
fig_h2.update_yaxes(title_text="Avg Profit (USD)", secondary_y=False)
fig_h2.update_yaxes(title_text="% Órdenes con pérdida", secondary_y=True)
fig_h2.show()

In [6]:
h3 = df.groupby(['State','Region']).agg(
    total_sales  =('Sales','sum'),
    total_profit =('Profit','sum'),
    margin_pct   =('Profit_Margin_pct','mean')
).reset_index()

avg_sales  = h3['total_sales'].mean()
avg_margin = h3['margin_pct'].mean()

def clasificar(row):
    if row['total_profit'] < 0:          return 'PERDIDA NETA'
    if row['total_sales'] > avg_sales and row['margin_pct'] > avg_margin:  return 'ESTRELLA'
    if row['total_sales'] > avg_sales and row['margin_pct'] <= avg_margin: return 'RIESGO'
    if row['total_sales'] <= avg_sales and row['margin_pct'] > avg_margin: return 'POTENCIAL'
    return 'CRITICO'

h3['clasificacion'] = h3.apply(clasificar, axis=1)
h3['size_val'] = h3['total_profit'].abs()

color_map = {
    'ESTRELLA'    : COLORS['alert_green'],
    'RIESGO'      : COLORS['alert_yell'],
    'POTENCIAL'   : COLORS['sales'],
    'CRITICO'     : '#FF8C00',
    'PERDIDA NETA': COLORS['alert_red']
}

fig_h3 = px.scatter(
    h3, x='total_sales', y='margin_pct',
    color='clasificacion', color_discrete_map=color_map,
    size='size_val', size_max=40,
    hover_name='State', hover_data=['Region','total_profit'],
    text='State',
    title='Cuadrantes de performance por Estado'
)
fig_h3.add_hline(y=avg_margin, line_dash='dash', line_color=COLORS['text_sec'],
                  annotation_text=f"Avg Margin: {avg_margin:.1f}%")
fig_h3.add_vline(x=avg_sales,  line_dash='dash', line_color=COLORS['text_sec'],
                  annotation_text=f"Avg Sales: $ {avg_sales:,.0f}")
fig_h3.update_traces(textposition='top center', textfont=dict(size=7))
fig_h3.update_layout(
    plot_bgcolor=COLORS['bg_canvas'],
    paper_bgcolor=COLORS['bg_card'],
    font=dict(family='Segoe UI'),
    height=500,
    xaxis=dict(title='Total Sales (USD)', gridcolor='#E0E0E0'),
    yaxis=dict(title='Profit Margin %', gridcolor='#E0E0E0')
)
fig_h3.show()

In [7]:
h5 = df.groupby(['Customer ID','Customer Name','Segment']).agg(
    total_sales =('Sales','sum'),
    total_profit=('Profit','sum')
).reset_index().sort_values('total_sales', ascending=False).reset_index(drop=True)

h5['pct_acum_sales']    = h5['total_sales'].cumsum() / h5['total_sales'].sum() * 100
h5['pct_clientes_acum'] = (h5.index + 1) / len(h5) * 100

corte_80 = h5[h5['pct_acum_sales'] <= 80].iloc[-1]

fig_h5 = make_subplots(specs=[[{"secondary_y": True}]])
fig_h5.add_trace(go.Bar(
    x=h5.index, y=h5['total_sales'],
    name='Sales por cliente',
    marker_color=COLORS['sales'],
    opacity=0.7
), secondary_y=False)
fig_h5.add_trace(go.Scatter(
    x=h5.index, y=h5['pct_acum_sales'],
    name='% Acumulado Sales',
    mode='lines',
    line=dict(color=COLORS['profit'], width=2)
), secondary_y=True)
fig_h5.add_hline(y=80, line_dash='dash', line_color=COLORS['alert_red'],
                  annotation_text="80%", secondary_y=True)
fig_h5.update_layout(
    title=dict(text=f"Pareto: el {corte_80['pct_clientes_acum']:.1f}% de clientes genera el 80% de ventas",
               font=dict(color=COLORS['text_title'], size=13)),
    plot_bgcolor=COLORS['bg_canvas'],
    paper_bgcolor=COLORS['bg_card'],
    font=dict(family='Segoe UI'),
    height=400,
    xaxis=dict(title='Clientes (ordenados por ventas)', gridcolor='#E0E0E0'),
    legend=dict(orientation='h', y=-0.2)
)
fig_h5.update_yaxes(title_text="Sales (USD)", secondary_y=False)
fig_h5.update_yaxes(title_text="% Acumulado", secondary_y=True, range=[0,105])
fig_h5.show()

In [8]:
h6 = df.groupby(['Order_Year','Order_Month']).agg(
    total_sales=('Sales','sum')
).reset_index()
h6_pivot = h6.pivot(index='Order_Month', columns='Order_Year', values='total_sales').fillna(0)

meses = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
         7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
h6_pivot.index = [meses[m] for m in h6_pivot.index]

fig_h6 = go.Figure(go.Heatmap(
    z=h6_pivot.values,
    x=[str(c) for c in h6_pivot.columns],
    y=h6_pivot.index,
    colorscale=[[0,'#FFFFFF'],[0.5,COLORS['sales']],[1,COLORS['profit']]],
    text=[[f"$ {v:,.0f}" for v in row] for row in h6_pivot.values],
    texttemplate="%{text}",
    textfont=dict(size=9),
    showscale=True
))
fig_h6.update_layout(
    title=dict(text="Heatmap de ventas: Mes × Año",
               font=dict(color=COLORS['text_title'], size=14)),
    plot_bgcolor=COLORS['bg_canvas'],
    paper_bgcolor=COLORS['bg_card'],
    font=dict(family='Segoe UI'),
    height=400,
    xaxis=dict(title='Año'),
    yaxis=dict(title='Mes', autorange='reversed')
)
fig_h6.show()

---
## 4. Dashboard Plotly Dash — Aplicación completa
> Réplica exacta del dashboard Power BI con las mismas 5 páginas y segmentaciones.  
> Para ejecutar: correr la celda y abrir http://127.0.0.1:8050 en el navegador.


In [9]:
app = dash.Dash(__name__, suppress_callback_exceptions=True)
app.title = "Dashboard de Ventas y Logística — Superstore"

anios    = sorted(df['Order_Year'].unique())
cats     = sorted(df['Category'].unique())
segs     = sorted(df['Segment'].unique())
regiones = sorted(df['Region'].unique())

FONT = 'Segoe UI, Arial, sans-serif'


def filtros_sidebar():
    label_style = {'color': COLORS['dona_2'], 'fontSize': '12px',
                   'fontFamily': FONT, 'fontWeight': '600'}
    return html.Div([
        html.H4("Filtros", style={
            'color': COLORS['bg_card'], 'fontFamily': FONT,
            'marginBottom': '16px', 'fontWeight': '700'
        }),
        html.Label("Año", style=label_style),
        dcc.Dropdown(
            id='filtro-anio',
            options=[{'label': str(a), 'value': a} for a in anios],
            multi=True, placeholder="Todos los años",
            style={'marginBottom': '12px', 'fontSize': '12px'}
        ),
        html.Label("Categoría", style=label_style),
        dcc.Dropdown(
            id='filtro-cat',
            options=[{'label': c, 'value': c} for c in cats],
            multi=True, placeholder="Todas las categorías",
            style={'marginBottom': '12px', 'fontSize': '12px'}
        ),
        html.Label("Segmento", style=label_style),
        dcc.Checklist(
            id='filtro-seg',
            options=[{'label': f" {s}", 'value': s} for s in segs],
            value=segs,
            labelStyle={'display': 'block', 'color': COLORS['bg_card'],
                        'fontFamily': FONT, 'fontSize': '12px',
                        'marginBottom': '4px'}
        ),
        html.Hr(style={'borderColor': COLORS['header'], 'margin': '16px 0'}),
        html.Label("Región", style=label_style),
        dcc.Checklist(
            id='filtro-region',
            options=[{'label': f" {r}", 'value': r} for r in regiones],
            value=regiones,
            labelStyle={'display': 'block', 'color': COLORS['bg_card'],
                        'fontFamily': FONT, 'fontSize': '12px',
                        'marginBottom': '4px'}
        ),
    ], style={
        'backgroundColor': COLORS['profit'],
        'padding': '20px',
        'borderRadius': '8px',
        'height': '100vh',
        'position': 'sticky',
        'top': '0',
        'overflowY': 'auto'
    })


app.layout = html.Div([
    html.Div([
        html.Div([
            html.H2("Dashboard de Ventas y Logística",
                    style={'color': COLORS['bg_card'], 'fontFamily': FONT,
                           'fontWeight': '700', 'margin': '0', 'fontSize': '20px'}),
            html.P("Superstore — Análisis Ejecutivo 2014-2017 | Python / Plotly Dash",
                   style={'color': COLORS['dona_2'], 'fontFamily': FONT,
                          'margin': '0', 'fontSize': '12px'})
        ]),
        html.P("Michael Hans Evans — Data Analyst",
               style={'color': COLORS['dona_2'], 'fontFamily': FONT,
                      'margin': '0', 'fontSize': '11px', 'alignSelf': 'center'})
    ], style={
        'backgroundColor': COLORS['profit'],
        'padding': '16px 24px',
        'display': 'flex',
        'justifyContent': 'space-between',
        'alignItems': 'center',
        'boxShadow': '0 2px 8px rgba(0,0,0,0.2)'
    }),

    html.Div([
        dcc.Tabs(id='tabs', value='portada', children=[
            dcc.Tab(label='Portada',              value='portada'),
            dcc.Tab(label='Resumen Ejecutivo',    value='resumen'),
            dcc.Tab(label='Rentabilidad',         value='rentabilidad'),
            dcc.Tab(label='Performance Regional', value='regional'),
            dcc.Tab(label='Clientes y Logística', value='clientes'),
            dcc.Tab(label='Estacionalidad',       value='estacionalidad'),
        ], colors={
            'border'    : COLORS['header'],
            'primary'   : COLORS['profit'],
            'background': COLORS['bg_canvas']
        }, style={'fontFamily': FONT, 'fontSize': '13px'})
    ], style={'padding': '0 24px', 'backgroundColor': COLORS['bg_canvas']}),

    html.Div([
        html.Div(filtros_sidebar(), style={'width': '220px', 'flexShrink': '0'}),
        html.Div(id='contenido-tab', style={
            'flex': '1',
            'padding': '16px',
            'overflowY': 'auto',
            'backgroundColor': COLORS['bg_canvas']
        })
    ], style={'display': 'flex', 'gap': '16px', 'padding': '16px',
              'backgroundColor': COLORS['bg_canvas'], 'minHeight': 'calc(100vh - 100px)'})

], style={'backgroundColor': COLORS['bg_canvas'], 'fontFamily': FONT})


def filtrar_df(anios_sel, cats_sel, segs_sel, regiones_sel):
    dff = df.copy()
    if anios_sel:
        dff = dff[dff['Order_Year'].isin(anios_sel)]
    if cats_sel:
        dff = dff[dff['Category'].isin(cats_sel)]
    if segs_sel:
        dff = dff[dff['Segment'].isin(segs_sel)]
    if regiones_sel:
        dff = dff[dff['Region'].isin(regiones_sel)]
    return dff


def base_layout(title, height=300, left_margin=140, showlegend=False):
    return dict(
        title=dict(text=title, font=dict(size=14, color=COLORS['text_title'], family=FONT)),
        plot_bgcolor='#F8F9FA',
        paper_bgcolor=COLORS['bg_card'],
        font=dict(family=FONT, size=11, color=COLORS['text_title']),
        margin=dict(l=left_margin, r=60, t=40, b=30),
        height=height,
        showlegend=showlegend,
        legend=dict(orientation='h', y=-0.15, font=dict(size=10)),
    )


def aplicar_ejes(fig):
    fig.update_xaxes(gridcolor='#E8E8E8', zeroline=True, zerolinecolor=COLORS['text_sec'])
    fig.update_yaxes(gridcolor='#E8E8E8')
    return fig


@app.callback(
    Output('contenido-tab', 'children'),
    [Input('tabs', 'value'),
     Input('filtro-anio', 'value'),
     Input('filtro-cat', 'value'),
     Input('filtro-seg', 'value'),
     Input('filtro-region', 'value')]
)
def render_tab(tab, anios_sel, cats_sel, segs_sel, regiones_sel):
    dff  = filtrar_df(anios_sel, cats_sel, segs_sel, regiones_sel)
    kpis = calcular_kpis(dff)

    grid = lambda cols: {'display': 'grid', 'gridTemplateColumns': f'repeat({cols}, 1fr)',
                         'gap': '12px', 'marginBottom': '16px'}

    if tab == 'portada':
        ordenes_perdida = int((dff['Profit'] < 0).sum())

        big_card = lambda t, v, c=COLORS['sales']: html.Div([
            html.P(t, style={
                'color': COLORS['text_sec'], 'fontSize': '14px',
                'fontFamily': FONT, 'margin': '0 0 8px 0', 'fontWeight': '600',
                'textTransform': 'uppercase', 'letterSpacing': '0.5px'
            }),
            html.H2(v, style={
                'color': c, 'fontFamily': FONT, 'fontSize': '36px',
                'fontWeight': '700', 'margin': '0'
            })
        ], style={**CARD_STYLE, 'padding': '28px 24px', 'textAlign': 'center'})

        def fmt_money(v): return f"$ {v:,.0f}"
        def fmt_pct(v): return f"{v:.2f}%"
        def fmt_num(v): return f"{int(v):,}"

        fila1 = html.Div([
            big_card("Total Registros",     fmt_num(kpis['total_orders'])),
            big_card("Total Customers",     fmt_num(kpis['total_customers'])),
            big_card("Total Sales",         fmt_money(kpis['total_sales'])),
            big_card("Total Profit",        fmt_money(kpis['total_profit']),
                     COLORS['alert_red'] if kpis['total_profit'] < 0 else COLORS['profit']),
        ], style={'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)',
                  'gap': '16px', 'marginBottom': '20px'})

        fila2 = html.Div([
            big_card("Profit Margin %",     fmt_pct(kpis['profit_margin'])),
            big_card("Avg Discount %",      fmt_pct(kpis['avg_discount'])),
            big_card("Órdenes con Pérdida", fmt_num(ordenes_perdida), COLORS['alert_red']),
            big_card("Avg Ticket",          fmt_money(kpis['avg_ticket'])),
        ], style={'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)',
                  'gap': '16px', 'marginBottom': '20px'})

        titulo = html.Div([
            html.H1("Dashboard de Ventas y Logística", style={
                'color': COLORS['bg_card'], 'fontFamily': FONT,
                'fontWeight': '700', 'margin': '0', 'fontSize': '28px'}),
            html.P("Superstore — Análisis Ejecutivo 2014-2017", style={
                'color': COLORS['dona_2'], 'fontFamily': FONT,
                'margin': '4px 0 0 0', 'fontSize': '16px'}),
        ], style={
            'backgroundColor': COLORS['header'], 'padding': '32px',
            'borderRadius': '8px', 'textAlign': 'center', 'marginBottom': '24px'
        })

        return html.Div([titulo, fila1, fila2], style={
            'display': 'flex', 'flexDirection': 'column',
            'justifyContent': 'center', 'minHeight': 'calc(100vh - 180px)'
        })

    elif tab == 'resumen':
        ordenes_perdida = int((dff['Profit'] < 0).sum())
        fila_kpis = html.Div([
            tarjeta_kpi("Profit Margin %",     kpis['profit_margin'], 'pct'),
            tarjeta_kpi("Total Profit",        kpis['total_profit'],  'moneda',
                        COLORS['alert_red'] if kpis['total_profit'] < 0 else COLORS['profit']),
            tarjeta_kpi("Total Registros",     kpis['total_orders']),
            tarjeta_kpi("Total Sales",         kpis['total_sales'],   'moneda'),
            tarjeta_kpi("Total Customers",     kpis['total_customers']),
            tarjeta_kpi("Avg Ticket",          kpis['avg_ticket'],    'moneda'),
            tarjeta_kpi("Órdenes con Pérdida", ordenes_perdida, 'numero', COLORS['alert_red']),
        ], style=grid(7))

        cat = dff.groupby('Category').agg(
            sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index().sort_values('sales')
        fig_cat = go.Figure()
        fig_cat.add_trace(go.Bar(
            y=cat['Category'], x=cat['sales'], orientation='h',
            name='Sales', marker_color=COLORS['sales'],
            text=cat['sales'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_cat.add_trace(go.Bar(
            y=cat['Category'], x=cat['profit'], orientation='h',
            name='Profit', marker_color=COLORS['profit'],
            text=cat['profit'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_cat.update_layout(**base_layout("Sales y Profit por Categoría", 300, 110, True))
        fig_cat.update_layout(barmode='group')
        fig_cat.update_xaxes(title=dict(text="USD", font=dict(size=11)))
        aplicar_ejes(fig_cat)

        cat['margin'] = (cat['profit'] / cat['sales'] * 100).round(2)
        fig_margin = go.Figure(go.Bar(
            y=cat['Category'], x=cat['margin'], orientation='h',
            marker_color=COLORS['sales'],
            text=cat['margin'].apply(lambda v: f"{v:.2f}%"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_margin.update_layout(**base_layout("Profit Margin % por Categoría", 300, 110))
        fig_margin.update_xaxes(title=dict(text="Margin %", font=dict(size=11)))
        aplicar_ejes(fig_margin)

        trim = dff.groupby(['Order_Year', 'Order_Quarter']).agg(
            sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index()
        trim['periodo'] = trim['Order_Year'].astype(int).astype(str) + '-Q' + trim['Order_Quarter'].astype(int).astype(str)
        trim = trim.sort_values(['Order_Year', 'Order_Quarter'])
        fig_tend = make_subplots(specs=[[{"secondary_y": True}]])
        fig_tend.add_trace(go.Scatter(
            x=trim['periodo'], y=trim['sales'], name='Sales',
            mode='lines+markers', line=dict(color=COLORS['sales'], width=3),
            marker=dict(size=7)), secondary_y=False)
        fig_tend.add_trace(go.Scatter(
            x=trim['periodo'], y=trim['profit'], name='Profit',
            mode='lines+markers', line=dict(color=COLORS['profit'], width=3),
            marker=dict(size=7)), secondary_y=True)
        fig_tend.update_layout(**base_layout("Tendencia Sales y Profit por Trimestre", 280, 70, True))
        fig_tend.update_xaxes(gridcolor='#E8E8E8', tickangle=-45, tickfont=dict(size=10))
        fig_tend.update_yaxes(title_text="Sales (USD)", gridcolor='#E8E8E8', secondary_y=False)
        fig_tend.update_yaxes(title_text="Profit (USD)", secondary_y=True)

        return html.Div([
            fila_kpis,
            html.Div([
                html.Div([dcc.Graph(figure=fig_cat)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_margin)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),
            html.Div([dcc.Graph(figure=fig_tend)], style=CARD_STYLE),
        ])

    elif tab == 'rentabilidad':
        ordenes_perdida = int((dff['Profit'] < 0).sum())
        fila_kpis = html.Div([
            tarjeta_kpi("Total Profit",        kpis['total_profit'],  'moneda',
                        COLORS['alert_red'] if kpis['total_profit'] < 0 else COLORS['profit']),
            tarjeta_kpi("Profit Margin %",     kpis['profit_margin'], 'pct'),
            tarjeta_kpi("Órdenes con Pérdida", ordenes_perdida, 'numero', COLORS['alert_red']),
            tarjeta_kpi("Avg Discount %",      kpis['avg_discount'],  'pct'),
        ], style=grid(4))

        sub = dff.groupby('Sub-Category').agg(
            profit=('Profit', 'sum')).reset_index().sort_values('profit')
        fig_sub = go.Figure(go.Bar(
            y=sub['Sub-Category'], x=sub['profit'], orientation='h',
            marker_color=[COLORS['alert_red'] if v < 0 else COLORS['sales'] for v in sub['profit']],
            text=sub['profit'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_sub.add_vline(x=0, line_color=COLORS['text_sec'], line_width=1)
        fig_sub.update_layout(**base_layout("Profit por Subcategoría", 400, 140))
        aplicar_ejes(fig_sub)

        dff2 = dff.copy()
        dff2['rango'] = pd.cut(dff2['Discount'],
            bins=[-0.01, 0, 0.20, 0.40, 0.60, 1.0],
            labels=['0%', '1-20%', '21-40%', '41-60%', '61-80%'])
        h2 = dff2.groupby('rango', observed=True).agg(
            avg_profit=('Profit', 'mean')).reset_index()
        fig_desc = go.Figure(go.Bar(
            y=h2['rango'], x=h2['avg_profit'], orientation='h',
            marker_color=[COLORS['alert_red'] if v < 0 else COLORS['sales'] for v in h2['avg_profit']],
            text=h2['avg_profit'].apply(lambda v: f"${v:.0f}"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_desc.add_vline(x=0, line_color=COLORS['text_sec'], line_width=1)
        fig_desc.update_layout(**base_layout("Avg Profit por rango de descuento", 400, 90))
        aplicar_ejes(fig_desc)

        loss = dff.groupby('Sub-Category').agg(
            perdidas=('Profit', lambda x: (x < 0).sum()),
            total=('Profit', 'count')).reset_index()
        loss['pct'] = (loss['perdidas'] / loss['total'] * 100).round(2)
        loss = loss[loss['pct'] > 0].sort_values('pct')
        fig_loss = go.Figure(go.Bar(
            y=loss['Sub-Category'], x=loss['pct'], orientation='h',
            marker_color=COLORS['sales'],
            text=loss['pct'].apply(lambda v: f"{v:.1f}%"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_loss.update_layout(**base_layout("% Órdenes con pérdida por Subcategoría", 400, 140))
        aplicar_ejes(fig_loss)

        sin = dff[dff['Discount'] == 0].groupby('Category')['Profit'].mean().round(2)
        con = dff[dff['Discount'] > 0].groupby('Category')['Profit'].mean().round(2)
        imp = pd.DataFrame({'Sin Descuento': sin, 'Con Descuento': con}).reset_index().sort_values('Sin Descuento')
        fig_impact = go.Figure()
        fig_impact.add_trace(go.Bar(
            y=imp['Category'], x=imp['Sin Descuento'], orientation='h',
            name='Sin Descuento', marker_color=COLORS['sales'],
            text=imp['Sin Descuento'].apply(lambda v: f"${v:.0f}"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_impact.add_trace(go.Bar(
            y=imp['Category'], x=imp['Con Descuento'], orientation='h',
            name='Con Descuento', marker_color=COLORS['profit'],
            text=imp['Con Descuento'].apply(lambda v: f"${v:.0f}"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_impact.add_vline(x=0, line_color=COLORS['text_sec'], line_width=1)
        fig_impact.update_layout(**base_layout("Impacto descuento por Categoría", 400, 110, True))
        fig_impact.update_layout(barmode='group')
        aplicar_ejes(fig_impact)

        return html.Div([
            fila_kpis,
            html.Div([
                html.Div([dcc.Graph(figure=fig_sub)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_desc)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),
            html.Div([
                html.Div([dcc.Graph(figure=fig_loss)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_impact)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px'}),
        ])

    elif tab == 'regional':
        central = dff[dff['Region'] == 'Central']
        cm = central['Profit'].sum() / central['Sales'].sum() * 100 if len(central) > 0 and central['Sales'].sum() > 0 else 0

        fila_kpis = html.Div([
            tarjeta_kpi("Total Sales",             kpis['total_sales'],   'moneda'),
            tarjeta_kpi("Profit Margin %",         kpis['profit_margin'], 'pct'),
            tarjeta_kpi("Total Profit",            kpis['total_profit'],  'moneda',
                        COLORS['alert_red'] if kpis['total_profit'] < 0 else COLORS['profit']),
            tarjeta_kpi("Profit Margin Central %", cm, 'pct'),
        ], style=grid(4))

        reg = dff.groupby('Region').agg(
            sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index().sort_values('sales')
        fig_reg = go.Figure()
        fig_reg.add_trace(go.Bar(
            y=reg['Region'], x=reg['sales'], orientation='h',
            name='Sales', marker_color=COLORS['sales'],
            text=reg['sales'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_reg.add_trace(go.Bar(
            y=reg['Region'], x=reg['profit'], orientation='h',
            name='Profit', marker_color=COLORS['profit'],
            text=reg['profit'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_reg.update_layout(**base_layout("Sales y Profit por Región", 300, 90, True))
        fig_reg.update_layout(barmode='group')
        aplicar_ejes(fig_reg)

        reg['margin'] = (reg['profit'] / reg['sales'] * 100).round(2)
        fig_reg_m = go.Figure(go.Bar(
            y=reg['Region'], x=reg['margin'], orientation='h',
            marker_color=COLORS['sales'],
            text=reg['margin'].apply(lambda v: f"{v:.2f}%"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_reg_m.update_layout(**base_layout("Profit Margin % por Región", 300, 90))
        aplicar_ejes(fig_reg_m)

        est = dff.groupby('State').agg(
            profit=('Profit', 'sum')).reset_index().sort_values('profit')
        fig_est = go.Figure(go.Bar(
            y=est['State'], x=est['profit'], orientation='h',
            marker_color=[COLORS['alert_red'] if v < 0 else COLORS['sales'] for v in est['profit']],
            text=est['profit'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_est.add_vline(x=0, line_color=COLORS['text_sec'], line_width=1)
        fig_est.update_layout(**base_layout("Ranking de Estados por Profit", max(400, len(est) * 18), 150))
        aplicar_ejes(fig_est)

        return html.Div([
            fila_kpis,
            html.Div([
                html.Div([dcc.Graph(figure=fig_reg)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_reg_m)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),
            html.Div([dcc.Graph(figure=fig_est)], style=CARD_STYLE),
        ])

    elif tab == 'clientes':
        cli_loss = dff.groupby('Customer ID')['Profit'].sum()
        clientes_perdida = int((cli_loss < 0).sum())
        pct_std = (dff['Ship Mode'] == 'Standard Class').mean() * 100

        fila_kpis = html.Div([
            tarjeta_kpi("Total Customers",      kpis['total_customers']),
            tarjeta_kpi("Avg Ticket",           kpis['avg_ticket'], 'moneda'),
            tarjeta_kpi("Clientes con Pérdida", clientes_perdida, 'numero', COLORS['alert_red']),
            tarjeta_kpi("Pct Standard Class",   pct_std, 'pct'),
        ], style=grid(4))

        top10 = dff.groupby(['Customer ID', 'Customer Name']).agg(
            sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index().nlargest(10, 'sales').sort_values('sales')
        fig_top = go.Figure()
        fig_top.add_trace(go.Bar(
            y=top10['Customer Name'], x=top10['sales'], orientation='h',
            name='Sales', marker_color=COLORS['sales'],
            text=top10['sales'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_top.add_trace(go.Bar(
            y=top10['Customer Name'], x=top10['profit'], orientation='h',
            name='Profit', marker_color=COLORS['profit'],
            text=top10['profit'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_top.update_layout(**base_layout("Top 10 Clientes por Total Sales", 400, 160, True))
        fig_top.update_layout(barmode='group')
        aplicar_ejes(fig_top)

        ship = dff.groupby('Ship Mode').size().reset_index(name='ordenes')
        fig_ship = go.Figure(go.Pie(
            labels=ship['Ship Mode'], values=ship['ordenes'], hole=0.4,
            marker_colors=[COLORS['sales'], COLORS['dona_2'], COLORS['profit'], COLORS['dona_4']],
            textinfo='label+percent', textfont=dict(size=11)))
        fig_ship.update_layout(
            title=dict(text="Distribución Ship Mode", font=dict(size=14, color=COLORS['text_title'], family=FONT)),
            paper_bgcolor=COLORS['bg_card'],
            font=dict(family=FONT, size=11, color=COLORS['text_title']),
            margin=dict(l=20, r=20, t=40, b=30), height=300,
            legend=dict(orientation='h', y=-0.1, font=dict(size=10)))

        cli_seg = dff.groupby(['Customer ID', 'Segment'])['Profit'].sum().reset_index()
        cli_seg_loss = cli_seg[cli_seg['Profit'] < 0].groupby('Segment').size().reset_index(name='count').sort_values('count')
        fig_cli_loss = go.Figure(go.Bar(
            y=cli_seg_loss['Segment'], x=cli_seg_loss['count'], orientation='h',
            marker_color=COLORS['sales'],
            text=cli_seg_loss['count'].apply(lambda v: f"{v:,}"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_cli_loss.update_layout(**base_layout("Clientes con pérdida por Segmento", 300, 110))
        aplicar_ejes(fig_cli_loss)

        sm_seg = dff.groupby(['Segment', 'Ship Mode']).size().reset_index(name='orders').sort_values('orders')
        ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
        sm_colors = [COLORS['sales'], COLORS['dona_2'], COLORS['profit'], COLORS['dona_4']]
        fig_ss = go.Figure()
        for i, sm in enumerate(ship_modes):
            d = sm_seg[sm_seg['Ship Mode'] == sm]
            fig_ss.add_trace(go.Bar(
                y=d['Segment'], x=d['orders'], orientation='h',
                name=sm, marker_color=sm_colors[i]))
        fig_ss.update_layout(**base_layout("Ship Mode por Segmento", 300, 110, True))
        fig_ss.update_layout(barmode='group')
        aplicar_ejes(fig_ss)

        return html.Div([
            fila_kpis,
            html.Div([
                html.Div([dcc.Graph(figure=fig_top)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_ship)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),
            html.Div([
                html.Div([dcc.Graph(figure=fig_cli_loss)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_ss)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px'}),
        ])

    elif tab == 'estacionalidad':
        q_sales = dff.groupby('Order_Quarter')['Sales'].sum()
        sales_q1 = q_sales.get(1, 0)
        sales_q4 = q_sales.get(4, 0)
        pct_q4 = sales_q4 / dff['Sales'].sum() * 100 if dff['Sales'].sum() > 0 else 0

        fila_kpis = html.Div([
            tarjeta_kpi("Total Sales",   kpis['total_sales'], 'moneda'),
            tarjeta_kpi("Sales Q1",      sales_q1,            'moneda'),
            tarjeta_kpi("Sales Q4",      sales_q4,            'moneda'),
            tarjeta_kpi("Pct Ventas Q4", pct_q4,              'pct'),
        ], style=grid(4))

        trim = dff.groupby(['Order_Year', 'Order_Quarter']).agg(
            sales=('Sales', 'sum'), profit=('Profit', 'sum')).reset_index()
        trim['periodo'] = trim['Order_Year'].astype(int).astype(str) + '-Q' + trim['Order_Quarter'].astype(int).astype(str)
        trim = trim.sort_values(['Order_Year', 'Order_Quarter'])
        fig_tend = make_subplots(specs=[[{"secondary_y": True}]])
        fig_tend.add_trace(go.Scatter(
            x=trim['periodo'], y=trim['sales'], name='Sales',
            mode='lines+markers', line=dict(color=COLORS['sales'], width=3),
            marker=dict(size=7)), secondary_y=False)
        fig_tend.add_trace(go.Scatter(
            x=trim['periodo'], y=trim['profit'], name='Profit',
            mode='lines+markers', line=dict(color=COLORS['profit'], width=3),
            marker=dict(size=7)), secondary_y=True)
        fig_tend.update_layout(**base_layout("Tendencia Sales y Profit por Año y Trimestre", 280, 70, True))
        fig_tend.update_xaxes(gridcolor='#E8E8E8', tickangle=-45, tickfont=dict(size=10))
        fig_tend.update_yaxes(title_text="Sales (USD)", gridcolor='#E8E8E8', secondary_y=False)
        fig_tend.update_yaxes(title_text="Profit (USD)", secondary_y=True)

        q_df = dff.groupby('Order_Quarter')['Sales'].sum().reset_index().sort_values('Sales')
        q_df['label'] = 'Q' + q_df['Order_Quarter'].astype(int).astype(str)
        fig_trim = go.Figure(go.Bar(
            y=q_df['label'], x=q_df['Sales'], orientation='h',
            marker_color=COLORS['sales'],
            text=q_df['Sales'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_trim.update_layout(**base_layout("Sales por Trimestre", 300, 80))
        aplicar_ejes(fig_trim)

        meses = {1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
                 7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'}
        m_df = dff.groupby('Order_Month')['Sales'].sum().reset_index().sort_values('Order_Month', ascending=False)
        m_df['label'] = m_df['Order_Month'].map(meses)
        fig_mes = go.Figure(go.Bar(
            y=m_df['label'], x=m_df['Sales'], orientation='h',
            marker_color=COLORS['sales'],
            text=m_df['Sales'].apply(lambda v: f"${v/1e3:.0f}K"),
            textposition='outside', textfont=dict(size=10, color=COLORS['text_title'])))
        fig_mes.update_layout(**base_layout("Sales por Mes", 400, 80))
        aplicar_ejes(fig_mes)

        return html.Div([
            fila_kpis,
            html.Div([dcc.Graph(figure=fig_tend)], style={**CARD_STYLE, 'marginBottom': '12px'}),
            html.Div([
                html.Div([dcc.Graph(figure=fig_trim)], style={**CARD_STYLE, 'flex': '1'}),
                html.Div([dcc.Graph(figure=fig_mes)], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex', 'gap': '12px'}),
        ])

    return html.Div("Selecciona una pestaña")

## 5. Ejecutar el dashboard

In [10]:
app.run(jupyter_mode='external', port=8050)

Dash app running on http://127.0.0.1:8050/


## 6. Validación — Consistencia con Power BI

In [11]:
kpis_totales = calcular_kpis(df)

pd.DataFrame({
    'KPI': ['Total Sales', 'Total Profit', 'Profit Margin %', 'Total Orders',
            'Total Customers', 'Avg Shipping Days', '% Órdenes con pérdida'],
    'Valor': [
        f"$ {kpis_totales['total_sales']:,.2f}",
        f"$ {kpis_totales['total_profit']:,.2f}",
        f"{kpis_totales['profit_margin']:.2f}%",
        f"{kpis_totales['total_orders']:,}",
        f"{kpis_totales['total_customers']:,}",
        f"{kpis_totales['avg_ship_days']:.1f}",
        f"{kpis_totales['pct_perdida']:.2f}%"
    ]
}).set_index('KPI')

,Valor
KPI,
Total Sales,"$ 2,296,919.49"
Total Profit,"$ 286,409.08"
Profit Margin %,12.47%
Total Orders,"9,993"
Total Customers,793
Avg Shipping Days,4.0
% Órdenes con pérdida,18.71%
